# Colab + Git LFS + local modules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/studiorabota/test-dg-2/blob/main/notebooks/windkit_demo.ipynb)

The **Open in Colab** button only loads this `.ipynb`. The first cells clone the repo and pull the Git LFS files (data + images), so the `windkit` package and the files are available in the runtime.

In [ ]:
# Repo this notebook belongs to
GITHUB_USER = "studiorabota"
REPO_NAME = "test-dg-2"
BRANCH = "main"

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    sh("git lfs version >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y git-lfs)")
    sh("git lfs install")

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")  # only needed for private repos
    except Exception:
        pass

    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{GITHUB_USER}/{REPO_NAME}.git"
    REPO_DIR = f"/content/{REPO_NAME}"

    if not os.path.isdir(REPO_DIR):
        result = subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR],
                                capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError("Clone failed. Check GITHUB_USER / REPO_NAME / BRANCH, "
                               "or add a GH_TOKEN secret for a private repo.")
    else:
        sh(f"git -C {REPO_DIR} pull --quiet")  # re-running: get the latest version
    sh(f"git -C {REPO_DIR} lfs pull")
else:
    REPO_DIR = os.path.abspath("..")  # running locally from notebooks/

# Make the windkit package importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo at:", REPO_DIR)

In [ ]:
# Files stored in Git LFS
sh(f"git -C {REPO_DIR} lfs ls-files --size")

## Load the data with the `windkit` module

In [ ]:
from pathlib import Path
import windkit as wk

data_path = Path(REPO_DIR) / "data" / "wind_speed_field.npy"
field = wk.load_field(data_path)  # raises a clear error if it's still an LFS pointer
wk.summary(field)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
wk.plot_mean_field(field, ax=ax1)
wk.plot_timeseries(field, ax=ax2)
plt.tight_layout()
plt.show()

In [ ]:
cf = wk.capacity_factor(field)
print(f"Mean capacity factor (generic 5 MW turbine): {cf.mean():.1%}")

## Images from the repo (also stored in LFS)

In [ ]:
from IPython.display import Image, display

images = sorted((Path(REPO_DIR) / "images").glob("*.png"))
for img in images:
    if wk.is_lfs_pointer(img):
        print(f"{img.name}: LFS pointer, not the image. Run `git lfs pull`.")
        continue
    print(f"{img.name} ({img.stat().st_size / 1e3:.0f} kB)")
    display(Image(filename=str(img), width=500))